# 🔧 Debugging Playbook — Common Failures & Fixes Across the AI Stack

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/00_debugging_playbook.ipynb)

A pattern-matching reference for diagnosing and fixing the most common failure modes
across all 8 domains of the Castalia AI engineering curriculum. Each entry follows
the same structure: **Symptom → Root Cause → Diagnosis → Fix**.

Keep this notebook open while you work through the course. When something breaks,
search here first.

## 📑 Table of Contents

1. [GPU & Memory Failures](#1-gpu--memory-failures)
2. [Model Loading Failures](#2-model-loading-failures)
3. [RAG Pipeline Failures](#3-rag-pipeline-failures)
4. [Agent Loop Failures](#4-agent-loop-failures)
5. [Finetuning Failures](#5-finetuning-failures)
6. [Eval Failures](#6-eval-failures)
7. [Prompt Engineering Failures](#7-prompt-engineering-failures)
8. [Systems & Serving Failures](#8-systems--serving-failures)
9. [General Diagnostic Toolkit](#9-general-diagnostic-toolkit)
10. [Key Takeaways & Quick Reference](#10-key-takeaways--quick-reference)

In [ ]:
# Shared imports used throughout the playbook
import sys, os, gc, time, json
from pathlib import Path

def safe_import(module_name):
    """Try to import a module; return None if unavailable."""
    try:
        return __import__(module_name)
    except ImportError:
        print(f"⚠️  {module_name} not installed")
        return None

torch = safe_import("torch")
transformers = safe_import("transformers")

---
## 1. GPU & Memory Failures

GPU issues are the single most common blocker in the course. Over half of all
"my code doesn't work" reports trace back to VRAM.

### 1.1 CUDA Out-of-Memory (OOM)

**Symptom:** `RuntimeError: CUDA out of memory. Tried to allocate X MiB ...`

**Root Cause:** The model, optimizer states, activations, and/or KV cache exceed
available VRAM. This is a hard limit — the GPU cannot swap to system RAM.

**Diagnosis Steps:**
1. Check current VRAM usage with `nvidia-smi` or `torch.cuda.memory_summary()`.
2. Estimate model size: `params × bytes_per_param` (fp16 = 2B/param, fp32 = 4B/param).
3. Account for overhead: optimizer states (2×–8× model size for Adam), activations,
   and KV cache grow with batch size and sequence length.

**Fixes (ordered by effort):**
| Fix | VRAM Savings | Tradeoff |
|-----|-------------|----------|
| Reduce batch size | Proportional | Slower training |
| Use `torch.float16` / `bfloat16` | ~50% | Minor precision loss |
| 4-bit quantization (bitsandbytes) | ~75% | Small quality drop |
| Gradient checkpointing | ~30-60% activations | ~20% slower |
| Manual cleanup: `del model; gc.collect(); torch.cuda.empty_cache()` | Varies | None |
| Use a smaller model | Proportional | Capability drop |

In [ ]:
# Diagnose CUDA OOM: check what is consuming VRAM
def diagnose_vram():
    """Print detailed VRAM breakdown for all visible GPUs."""
    if not torch or not torch.cuda.is_available():
        print("❌ No CUDA GPU available")
        return

    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        total = props.total_mem / 1024**3
        alloc = torch.cuda.memory_allocated(i) / 1024**3
        reserved = torch.cuda.memory_reserved(i) / 1024**3
        free = total - reserved
        print(f"GPU {i}: {props.name}")
        print(f"  Total:    {total:.1f} GB")
        print(f"  Allocated:{alloc:.1f} GB")
        print(f"  Reserved: {reserved:.1f} GB")
        print(f"  Free:     {free:.1f} GB")
        print()

    # Show full memory summary for the current device
    print(torch.cuda.memory_summary(abbreviated=True))

diagnose_vram()

In [ ]:
# Emergency VRAM cleanup — run this after OOM before retrying
def emergency_cleanup():
    """Aggressively free GPU memory."""
    gc.collect()
    if torch and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        alloc = torch.cuda.memory_allocated() / 1024**3
        print(f"✅ Cache cleared. Remaining allocated: {alloc:.2f} GB")
    else:
        print("No CUDA device — nothing to clear.")

emergency_cleanup()

### 1.2 "No GPU Runtime"

**Symptom:** `torch.cuda.is_available()` returns `False`, or `nvidia-smi` shows
"command not found" / no devices.

**Root Cause:**
- Colab: Runtime type is set to CPU (default). GPU was not requested or quota exhausted.
- Local: NVIDIA drivers not installed, or PyTorch installed without CUDA support.

**Diagnosis:**
```python
import torch
print(torch.cuda.is_available())          # Should be True
print(torch.version.cuda)                 # Should show CUDA version
!nvidia-smi                               # Should list GPU(s)
```

**Fix (Colab):** Runtime → Change runtime type → **T4 GPU** → Save → restart.

**Fix (local):** Install the CUDA-enabled PyTorch:
```bash
pip install torch --index-url https://download.pytorch.org/whl/cu121
```

### 1.3 Slow Inference

**Symptom:** Generation takes minutes for a short response. Tokens/second is far
below expected throughput for the hardware.

**Root Causes (in order of likelihood):**
1. **Model is on CPU.** Check `model.device` — if it says `cpu`, move it: `model.to("cuda")`.
2. **KV cache not enabled.** Transformers `generate()` uses KV cache by default, but
   custom loops may recompute attention every step.
3. **Excessive padding.** Long padding inflates compute. Use `padding_side="left"` and
   trim to actual sequence length.
4. **Synchronous generation.** Use batching or a serving framework (vLLM, SGLang)
   for continuous batching.
5. **Compilation not used.** `torch.compile(model)` can give 10-30% speedup on modern GPUs.

In [ ]:
# Quick inference profiler
def profile_generation(model, tokenizer, prompt="Hello, world!", max_new_tokens=100):
    """Measure tokens/second for a single generation."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    # Warmup
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=1)

    # Timed run
    if torch and torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    if torch and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    gen_tokens = out.shape[1] - input_len
    tok_per_sec = gen_tokens / elapsed
    print(f"Generated {gen_tokens} tokens in {elapsed:.2f}s → {tok_per_sec:.1f} tok/s")
    print(f"Model device: {next(model.parameters()).device}")
    return tok_per_sec

---
## 2. Model Loading Failures

### 2.1 Model Too Large for VRAM

**Symptom:** OOM during `from_pretrained()` or the process is killed by the OS
(exit code 137 / SIGKILL).

**Root Cause:** Model parameters in fp16 exceed VRAM. A 7B-param model needs ~14 GB
in fp16 and ~7 GB in 4-bit quantization. T4 GPUs have only 15 GB.

**Quick VRAM estimation:**
| Model Size | fp32 | fp16/bf16 | 8-bit | 4-bit |
|-----------|------|-----------|-------|-------|
| 1B params | 4 GB | 2 GB | 1 GB | 0.5 GB |
| 7B params | 28 GB | 14 GB | 7 GB | 3.5 GB |
| 13B params | 52 GB | 26 GB | 13 GB | 6.5 GB |
| 70B params | 280 GB | 140 GB | 70 GB | 35 GB |

**Fix:** Load with quantization:
```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
```

### 2.2 Tokenizer Mismatch

**Symptom:** Model outputs garbage, starts mid-word, or produces repeated special tokens.
Chat models output raw text instead of following the expected conversation format.

**Root Causes:**
- Using the wrong tokenizer (e.g., base model tokenizer with a chat-finetuned model).
- Missing chat template: `tokenizer.chat_template` is `None` or wrong.
- Special tokens (`bos_token`, `eos_token`, `pad_token`) not set correctly.

**Diagnosis:**
```python
print(tokenizer.chat_template)             # Should not be None for chat models
print(tokenizer.special_tokens_map)         # Check bos, eos, pad
print(tokenizer.apply_chat_template(msgs, tokenize=False))  # Inspect formatted prompt
```

**Fix:** Always load tokenizer from the same model ID. For chat models, verify the
template produces the expected format before running inference.

In [ ]:
# Model loading diagnostic
def diagnose_model_loading(model_id, max_vram_gb=None):
    """
    Test whether a model can be loaded. Reports VRAM requirements
    and suggests quantization if needed.
    """
    from transformers import AutoConfig

    print(f"🔍 Checking: {model_id}")
    print("=" * 50)

    # Get model config without downloading weights
    try:
        config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    except Exception as e:
        print(f"❌ Cannot fetch config: {e}")
        return

    # Estimate parameter count
    vocab = getattr(config, "vocab_size", 32000)
    hidden = getattr(config, "hidden_size", 4096)
    layers = getattr(config, "num_hidden_layers", 32)
    intermediate = getattr(config, "intermediate_size", hidden * 4)
    # Rough estimate: embedding + layers × (attn + ffn)
    est_params = vocab * hidden + layers * (4 * hidden**2 + 3 * hidden * intermediate)
    est_params_b = est_params / 1e9

    fp16_gb = est_params_b * 2
    int4_gb = est_params_b * 0.5
    print(f"  Estimated params: {est_params_b:.1f}B")
    print(f"  VRAM (fp16):      {fp16_gb:.1f} GB")
    print(f"  VRAM (4-bit):     {int4_gb:.1f} GB")

    if max_vram_gb:
        if fp16_gb > max_vram_gb and int4_gb <= max_vram_gb:
            print(f"  ⚠️  fp16 won't fit in {max_vram_gb} GB — use 4-bit quantization")
        elif int4_gb > max_vram_gb:
            print(f"  ❌ Even 4-bit won't fit in {max_vram_gb} GB — use a smaller model")
        else:
            print(f"  ✅ Fits in {max_vram_gb} GB at fp16")

# Example: diagnose_model_loading("meta-llama/Llama-3.1-8B-Instruct", max_vram_gb=15)

---
## 3. RAG Pipeline Failures

RAG has many moving parts: chunking → embedding → indexing → retrieval → prompt
assembly → generation. A bug in any stage silently degrades the final answer.

### 3.1 Retrieved Chunks Are Irrelevant

**Symptom:** The model's answer is generic or wrong, even though the correct information
exists in the corpus.

**Root Causes:**
1. **Embedding model mismatch.** Query and document embeddings use different models
   or different preprocessing (e.g., one normalizes, the other doesn't).
2. **Chunks too large.** A 2000-token chunk buries the relevant sentence in noise.
   Cosine similarity becomes diluted.
3. **No metadata filtering.** Retrieving from the entire corpus when the question
   targets a specific document or section.
4. **Wrong similarity metric.** Using L2 distance with non-normalized embeddings, or
   cosine similarity when the model was trained with dot product.

**Diagnosis:** Always inspect the actual retrieved chunks before blaming the LLM.
Print them. Read them. Are they relevant to the query?

**Fix:** Use 256–512 token chunks with 10–20% overlap. Match the similarity metric
to what the embedding model was trained with. Add metadata filters.

### 3.2 Hallucination Despite RAG

**Symptom:** The model generates confident but incorrect information, even though
relevant context was retrieved.

**Root Causes:**
1. **Context not actually in the prompt.** The retrieval step found good chunks, but
   they weren't inserted into the prompt (a wiring bug).
2. **Context window overflow.** The prompt + context exceeds the model's max length,
   and the context gets silently truncated.
3. **Model ignores context.** The system prompt doesn't instruct the model to answer
   based on the provided context. Or the context is placed after the question, where
   some models attend to it less.

**Fix:** Count tokens. Put context before the question. Add an explicit instruction:
"Answer ONLY based on the following context. If the answer is not in the context,
say 'I don't know'."

### 3.3 FAISS Index Errors

**Symptom:** `RuntimeError: Sizes of tensors must match` or empty search results
from a non-empty index.

**Root Causes:**
- **Dimension mismatch:** Query embedding dimension ≠ index dimension. Happens when
  you rebuild the index with a different embedding model but use old query code.
- **Empty index:** The index was created but never populated (`index.ntotal == 0`).
- **Serialization bug:** Loaded an index from disk that was built with a different
  FAISS version or different parameters.

**Diagnosis:**
```python
print(f"Index dimension: {index.d}")
print(f"Index size: {index.ntotal}")
print(f"Query dimension: {query_embedding.shape}")
```

In [ ]:
# RAG pipeline diagnostic — check each stage
def diagnose_rag_pipeline(query, chunks, embeddings, index, embed_fn, top_k=5):
    """
    Walk through each RAG stage and report what might be wrong.

    Args:
        query: The user question (str)
        chunks: List of text chunks in the corpus
        embeddings: numpy array of chunk embeddings (N x D)
        index: FAISS index object
        embed_fn: Function that takes a string and returns a numpy vector
        top_k: Number of chunks to retrieve
    """
    import numpy as np

    print("🔍 RAG Pipeline Diagnostic")
    print("=" * 50)

    # Stage 1: Corpus
    print(f"\n📚 Corpus: {len(chunks)} chunks")
    if len(chunks) == 0:
        print("  ❌ Corpus is empty!")
        return
    lengths = [len(c.split()) for c in chunks]
    print(f"  Word counts — min: {min(lengths)}, max: {max(lengths)}, "
          f"mean: {np.mean(lengths):.0f}")
    if max(lengths) > 500:
        print("  ⚠️  Some chunks are very large — consider smaller chunk sizes")

    # Stage 2: Embeddings
    print(f"\n🔢 Embeddings shape: {embeddings.shape}")
    if embeddings.shape[0] != len(chunks):
        print(f"  ❌ Mismatch: {embeddings.shape[0]} embeddings for {len(chunks)} chunks")

    # Stage 3: Index
    print(f"\n📇 Index — dimension: {index.d}, vectors: {index.ntotal}")
    if index.ntotal == 0:
        print("  ❌ Index is empty — did you call index.add()?")
        return
    if index.d != embeddings.shape[1]:
        print(f"  ❌ Index dim ({index.d}) ≠ embedding dim ({embeddings.shape[1]})")

    # Stage 4: Query embedding + retrieval
    q_emb = embed_fn(query)
    print(f"\n🔎 Query embedding shape: {q_emb.shape}")
    if q_emb.shape[-1] != index.d:
        print(f"  ❌ Query dim ({q_emb.shape[-1]}) ≠ index dim ({index.d})")
        return

    q_emb_2d = q_emb.reshape(1, -1).astype("float32")
    distances, indices = index.search(q_emb_2d, top_k)
    print(f"  Top-{top_k} distances: {distances[0]}")
    print(f"  Top-{top_k} indices:   {indices[0]}")

    # Stage 5: Inspect retrieved chunks
    print(f"\n📝 Retrieved chunks:")
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        preview = chunks[idx][:150].replace("\n", " ")
        print(f"  [{rank+1}] (idx={idx}, dist={dist:.4f}) {preview}...")

    print("\n✅ Pipeline looks wired correctly. Check chunk content for relevance.")

---
## 4. Agent Loop Failures

Agents combine LLM reasoning with tool use in a loop. When they break, the failure
mode is often subtle: the agent doesn't crash — it just loops, halllucinates tool
calls, or ignores results.

### 4.1 Agent Loops Forever

**Symptom:** The agent keeps calling tools or regenerating the same response without
converging on a final answer. Token costs explode.

**Root Causes:**
1. **Missing stop condition.** The loop has no `max_iterations` guard or the LLM
   never generates the stop token/phrase.
2. **Tool always fails.** The tool returns an error every time, and the agent retries
   without a fallback strategy.
3. **Ambiguous instructions.** The system prompt doesn't clearly define when the agent
   should stop and produce a final answer.

**Fix:**
- Always set `max_iterations` (5–10 is usually enough).
- Add a fallback: "If a tool fails twice, answer with what you know."
- Make the stop condition explicit: "When you have the final answer, respond with
  FINAL ANSWER: <answer>" or use a structured output schema with a `done` field.

### 4.2 Tool Call Parsing Errors

**Symptom:** `JSONDecodeError`, `KeyError`, or the tool receives garbage arguments.

**Root Causes:**
- The LLM generates malformed JSON (trailing comma, unescaped quotes, markdown fences).
- Schema mismatch: the LLM was told the tool has different parameters than it actually has.
- The prompt doesn't include few-shot examples of correct tool call format.

**Fix:**
- Use structured output / function calling APIs when available (OpenAI, vLLM).
- If parsing raw text, use a lenient JSON parser that strips markdown fences:
  ```python
  import re, json
  text = re.sub(r"```json?\n?|```", "", raw_output)
  parsed = json.loads(text.strip())
  ```
- Validate tool calls against the schema before execution.

### 4.3 Agent Ignores Tool Results

**Symptom:** The agent calls a tool, gets a correct result, but then answers as if
the tool was never called.

**Root Causes:**
- **Context too long.** The conversation history (including previous tool results)
  exceeds the model's effective attention window. Recent results get lost.
- **Tool results not formatted clearly.** The result is a raw dump that the model
  can't easily parse. Wrap results in clear delimiters.
- **System prompt override.** The model's prior knowledge overrides the tool result
  because the prompt doesn't emphasize trusting tool outputs.

**Fix:** Summarize conversation history. Format tool results as:
```
[Tool Result: search]
Found 3 results:
1. ...
2. ...
```
Add to system prompt: "Always use tool results to inform your answer."

In [ ]:
# Agent trace debugger — log each iteration of an agent loop
class AgentTracer:
    """Wraps an agent loop to log every step for debugging."""

    def __init__(self, max_iterations=10):
        self.max_iterations = max_iterations
        self.trace = []

    def log_step(self, iteration, thought, tool_call=None, tool_result=None):
        step = {
            "iteration": iteration,
            "thought": thought[:200],  # Truncate for readability
            "tool_call": tool_call,
            "tool_result": str(tool_result)[:300] if tool_result else None,
        }
        self.trace.append(step)

    def detect_loops(self):
        """Check if the agent is repeating the same tool calls."""
        if len(self.trace) < 3:
            return False
        recent_calls = [s.get("tool_call") for s in self.trace[-3:]]
        if len(set(json.dumps(c, sort_keys=True) for c in recent_calls if c)) == 1:
            return True
        return False

    def print_trace(self):
        """Pretty-print the full agent trace."""
        print("\n🔄 Agent Trace")
        print("=" * 60)
        for step in self.trace:
            i = step["iteration"]
            print(f"\n--- Step {i} ---")
            print(f'  Thought: {step["thought"]}')
            if step["tool_call"]:
                print(f'  Tool:    {json.dumps(step["tool_call"])}')
                print(f'  Result:  {step["tool_result"]}')
        if self.detect_loops():
            print("\n⚠️  WARNING: Agent appears to be in a loop "
                  "(last 3 tool calls are identical)")

# Usage:
# tracer = AgentTracer(max_iterations=10)
# for i in range(tracer.max_iterations):
#     thought, tool_call = agent_step(messages)
#     result = execute_tool(tool_call)
#     tracer.log_step(i, thought, tool_call, result)
#     if tracer.detect_loops():
#         print("Breaking: loop detected")
#         break
# tracer.print_trace()

---
## 5. Finetuning Failures

Finetuning failures are insidious because the training loop may complete without
errors, but the resulting model is degraded. Always compare against the base model.

### 5.1 Loss Doesn't Decrease

**Symptom:** Training loss stays flat or oscillates wildly from the first step.

**Root Causes:**
1. **Learning rate too high.** Loss oscillates or increases. Start with 1e-5 to 2e-4
   for LoRA finetuning.
2. **Learning rate too low.** Loss decreases imperceptibly. Try 10× higher.
3. **Data format wrong.** The model is trained on garbled input because the chat
   template wasn't applied, or special tokens are missing.
4. **Loss masking issue.** Loss is computed on the prompt tokens instead of (or in
   addition to) the completion tokens. The model learns to predict the prompt, which
   is trivially easy, masking the real loss.

**Diagnosis:** Print 3-5 tokenized training examples and decode them. Verify the
labels mask the prompt tokens (labels = -100 for prompt positions).

### 5.2 Loss Spikes

**Symptom:** Loss suddenly jumps to a very high value mid-training, then may or may
not recover.

**Root Causes:**
- **Gradient explosion.** Check for `nan` or `inf` in gradients. Use gradient clipping
  (`max_grad_norm=1.0`).
- **Bad data samples.** A corrupted or extremely unusual sample causes a loss spike.
  Log the batch contents at each step to identify the culprit.
- **Learning rate warmup missing.** Large initial gradients cause instability. Use
  linear warmup for the first 5-10% of steps.

**Fix:** Enable `max_grad_norm=1.0`, add warmup steps, and inspect data quality
by sorting samples by loss (highest-loss samples are often malformed).

### 5.3 Finetuned Model Worse Than Base

**Symptom:** The finetuned model scores lower on evals than the base model, or
generates incoherent text.

**Root Causes:**
1. **Catastrophic forgetting.** Too many steps or too high a learning rate destroyed
   the model's general knowledge. LoRA with low rank (r=8–16) mitigates this.
2. **Overfitting.** The model memorized the training set. Check: does training loss
   decrease while validation loss increases?
3. **Wrong evaluation.** You're evaluating with a different prompt format than the
   model was finetuned on. Always use the exact same chat template.
4. **Merged incorrectly.** LoRA adapters merged with wrong base model or wrong merge
   method.

**Fix:** Use LoRA instead of full finetuning. Train for fewer steps (1-3 epochs).
Always evaluate on a held-out set with the correct prompt format.

In [ ]:
# Finetuning diagnostic: check data format and loss masking
def diagnose_finetuning_data(dataset, tokenizer, num_samples=3):
    """
    Inspect tokenized training examples to verify formatting and loss masking.
    """
    print("🔍 Finetuning Data Diagnostic")
    print("=" * 50)

    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        input_ids = sample["input_ids"]
        labels = sample.get("labels", input_ids)

        # Decode full sequence
        full_text = tokenizer.decode(input_ids, skip_special_tokens=False)
        print(f"\n--- Sample {i} ---")
        print(f"  Length: {len(input_ids)} tokens")
        print(f"  Text preview: {full_text[:200]}...")

        # Check loss masking
        if hasattr(labels, "__len__"):
            masked = sum(1 for l in labels if l == -100)
            trained = len(labels) - masked
            print(f"  Labels: {masked} masked (prompt), {trained} trained (completion)")
            if trained == 0:
                print("  ❌ All labels are masked — model learns nothing!")
            elif masked == 0:
                print("  ⚠️  No labels masked — model trains on prompt too")
            else:
                # Show where the split is
                split_idx = next((j for j, l in enumerate(labels) if l != -100), 0)
                prompt_text = tokenizer.decode(input_ids[:split_idx])
                completion_text = tokenizer.decode(input_ids[split_idx:split_idx+50])
                print(f"  Prompt ends with: ...{prompt_text[-80:]}")
                print(f"  Completion starts: {completion_text[:80]}...")

    # Check for common issues
    print("\n📊 Dataset stats:")
    print(f"  Total samples: {len(dataset)}")
    if len(dataset) < 100:
        print("  ⚠️  Very small dataset — high risk of overfitting")

# Usage: diagnose_finetuning_data(tokenized_dataset, tokenizer)

---
## 6. Eval Failures

Bad evaluation is worse than no evaluation — it gives you false confidence. Most
eval bugs produce numbers that look reasonable but measure the wrong thing.

### 6.1 Metrics Disagree

**Symptom:** Model A wins on BLEU but loses on human preference. Precision is high
but recall is low. Faithfulness is perfect but relevance is terrible.

**Root Cause:** Different metrics measure fundamentally different things:

| Metric | What it measures | When it misleads |
|--------|-----------------|------------------|
| BLEU / ROUGE | N-gram overlap with reference | Penalizes valid paraphrases |
| Exact Match | Character-identical | Too strict for open-ended tasks |
| Precision@k | How many retrieved items are relevant | Ignores missed relevant items |
| Recall@k | How many relevant items were found | Ignores irrelevant noise |
| Faithfulness | Does output match the source | Doesn't measure usefulness |
| Helpfulness | Is the output useful | Subjective, hard to automate |

**Fix:** Choose metrics that align with your actual goal. For RAG, faithfulness +
relevance. For chat, human preference or LLM-as-judge with a clear rubric. Always
report multiple metrics — don't cherry-pick the flattering one.

### 6.2 LLM-as-Judge Inconsistency

**Symptom:** Re-running the same evaluation produces different scores. The judge
gives 5/5 to clearly bad outputs, or 1/5 to good ones.

**Root Causes:**
- **Temperature too high.** Use `temperature=0` for deterministic evaluation.
- **Rubric is ambiguous.** "Rate the quality on 1-5" is too vague. Define each
  score level explicitly.
- **Position bias.** When comparing two outputs, the judge tends to prefer whichever
  appears first. Randomize order and average.
- **Self-bias.** Models tend to rate their own outputs higher. Use a different model
  as judge than the one being evaluated.

**Fix:**
```python
# Good rubric example
rubric = """
Score 1: Completely wrong or irrelevant answer.
Score 2: Partially relevant but contains major errors.
Score 3: Mostly correct but missing important details.
Score 4: Correct and complete, minor issues only.
Score 5: Perfect — correct, complete, well-structured.
"""
```

### 6.3 Eval Set Too Small

**Symptom:** A "3% improvement" that reverses direction every time you re-evaluate
on a different subset.

**Root Cause:** With fewer than ~100 samples, random variation dominates. A single
outlier can swing the score by 5-10%.

**Diagnosis:** Compute confidence intervals. If the 95% CI is wider than the
difference you're trying to detect, your eval set is too small.

**Fix:** Use at least 100 samples for rough comparisons, 300+ for reliable results.
Report confidence intervals, not just point estimates.

**Rule of thumb:** To detect a `d`% difference with 95% confidence, you need
roughly `(1/d)² × 400` samples. So a 5% difference requires ~1,600 samples.

In [ ]:
# Eval sanity checker
import math

def eval_sanity_check(scores, labels=None, confidence=0.95):
    """
    Check whether your eval results are statistically meaningful.

    Args:
        scores: list of numeric scores (e.g., per-sample accuracy: 0 or 1)
        labels: optional list of labels for stratified analysis
        confidence: confidence level for interval
    """
    n = len(scores)
    mean = sum(scores) / n
    variance = sum((s - mean) ** 2 for s in scores) / (n - 1) if n > 1 else 0
    std = math.sqrt(variance)
    se = std / math.sqrt(n)

    # Z-score for 95% confidence
    z = 1.96 if confidence == 0.95 else 2.576
    ci_low = mean - z * se
    ci_high = mean + z * se

    print("📊 Eval Sanity Check")
    print("=" * 40)
    print(f"  Samples:     {n}")
    print(f"  Mean score:  {mean:.4f}")
    print(f"  Std dev:     {std:.4f}")
    print(f"  {int(confidence*100)}% CI:      [{ci_low:.4f}, {ci_high:.4f}]")
    print(f"  CI width:    {ci_high - ci_low:.4f}")

    if n < 30:
        print("  ❌ Fewer than 30 samples — results are unreliable")
    elif n < 100:
        print("  ⚠️  Fewer than 100 samples — treat results as directional only")
    else:
        detectable = 2 * z * se  # Minimum detectable difference
        print(f"  ✅ Minimum detectable difference: {detectable:.4f}")

    return {"mean": mean, "std": std, "ci": (ci_low, ci_high), "n": n}

# Example: eval_sanity_check([1, 0, 1, 1, 0, 1, 1, 1, 0, 1] * 10)

---
## 7. Prompt Engineering Failures

### 7.1 Model Doesn't Follow Instructions

**Symptom:** The model ignores parts of the prompt, gives the wrong format, or
does something completely different from what was asked.

**Root Causes:**
1. **Prompt too complex.** Multiple instructions competing for attention. The model
   follows some and ignores others.
2. **Wrong format for the model.** Using ChatML format with a model trained on Llama
   format, or vice versa. The model doesn't recognize the structure.
3. **Instruction buried in context.** The key instruction is sandwiched between large
   blocks of context. Put critical instructions at the start and end of the prompt.
4. **Model too small.** Small models (< 7B) struggle with complex multi-step instructions.

**Fix:** Simplify. One clear instruction per prompt. Use the model's native chat template.
Put the most important instruction last (recency bias). Test with a trivial example
first to isolate prompt issues from capability issues.

### 7.2 Inconsistent Outputs

**Symptom:** The same prompt produces wildly different outputs each time. Results
are non-reproducible.

**Root Causes:**
- **Temperature too high.** `temperature=1.0` (default for many APIs) adds significant
  randomness. Use `temperature=0` or `0.1` for deterministic tasks.
- **No seed set.** Even with low temperature, different random seeds produce different
  top-k/top-p sampling paths.
- **Non-deterministic infrastructure.** Different GPU types, different batch sizes, or
  floating-point non-determinism can change results.

**Fix:**
```python
response = client.chat.completions.create(
    model="...",
    messages=messages,
    temperature=0,     # Deterministic
    seed=42,           # Reproducible (when supported)
)
```

### 7.3 Token Limit Exceeded

**Symptom:** Output is truncated mid-sentence. Or you get an API error:
`context_length_exceeded`. The model starts repeating itself near the end.

**Root Cause:** `prompt_tokens + max_new_tokens > model_context_window`. For a
4K-context model with a 3.5K prompt, you can only generate 500 tokens.

**Diagnosis:** Count tokens before sending:
```python
token_count = len(tokenizer.encode(prompt))
print(f"Prompt: {token_count} tokens, budget: {context_window - token_count} remaining")
```

**Fix:** Shorten the prompt (summarize context, remove examples), use a model with
a larger context window, or split the task across multiple calls.

In [ ]:
# Prompt debugger: token count, format check, template validation
def debug_prompt(messages, tokenizer=None, model_context_window=4096,
                 max_new_tokens=512):
    """
    Analyze a prompt for common issues before sending to a model.

    Args:
        messages: list of {"role": ..., "content": ...} dicts
        tokenizer: HuggingFace tokenizer (optional, for token counting)
        model_context_window: model's max context length
        max_new_tokens: planned max generation length
    """
    print("🔍 Prompt Debugger")
    print("=" * 50)

    # Message structure
    roles = [m["role"] for m in messages]
    print(f"  Messages: {len(messages)} ({', '.join(roles)})")

    if roles[0] != "system" and len(messages) > 1:
        print("  ⚠️  No system message — consider adding one")
    if roles[-1] != "user":
        print("  ⚠️  Last message is not from user — model may be confused")

    # Content lengths
    for i, m in enumerate(messages):
        chars = len(m["content"])
        words = len(m["content"].split())
        print(f'  [{i}] {m["role"]:>10}: {words} words, {chars} chars')

    # Token count (if tokenizer available)
    if tokenizer:
        if hasattr(tokenizer, "apply_chat_template"):
            formatted = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            token_count = len(tokenizer.encode(formatted))
        else:
            text = "\n".join(m["content"] for m in messages)
            token_count = len(tokenizer.encode(text))

        budget = model_context_window - token_count
        print(f"\n  Token count:  {token_count}")
        print(f"  Context window: {model_context_window}")
        print(f"  Budget for generation: {budget}")
        print(f"  Planned max_new_tokens: {max_new_tokens}")

        if budget <= 0:
            print("  ❌ Prompt already exceeds context window!")
        elif budget < max_new_tokens:
            print(f"  ⚠️  Only {budget} tokens left — output will be truncated")
        else:
            print("  ✅ Enough room for generation")

        # Check chat template
        if hasattr(tokenizer, "chat_template") and tokenizer.chat_template:
            print(f"\n  Chat template: {tokenizer.chat_template[:80]}...")
        else:
            print("\n  ⚠️  No chat template set on tokenizer")

# Example:
# debug_prompt(
#     [{"role": "system", "content": "You are helpful."},
#      {"role": "user", "content": "Explain transformers."}],
#     tokenizer=tokenizer
# )

---
## 8. Systems & Serving Failures

Serving is where "it works on my machine" meets reality. The failure modes are
different from training and research — they're about latency, throughput, and
compatibility.

### 8.1 High Latency

**Symptom:** Time-to-first-token (TTFT) is seconds, not milliseconds. End-to-end
latency is 10× worse than expected for the hardware.

**Root Causes:**
1. **KV cache disabled.** Each token regenerates attention for all previous tokens.
   Inference is O(n²) instead of O(n). In vLLM and SGLang, KV cache is automatic.
   In raw Transformers, ensure `use_cache=True` (the default).
2. **Model not quantized.** fp32 inference uses 4× the memory bandwidth of int8.
   Bandwidth is usually the bottleneck for autoregressive generation.
3. **Wrong batch size.** Batch size 1 underutilizes the GPU. But too-large batches
   cause KV cache thrashing. Profile to find the sweet spot.
4. **Prefix caching not enabled.** If many requests share the same system prompt,
   enabling prefix caching avoids redundant prefill computation.
5. **Long input, short output.** Prefill (processing the prompt) dominates. Use
   chunked prefill to overlap prefill with decoding for other requests.

**Diagnosis:**
```bash
# Measure TTFT and total latency
curl -w "\nTTFB: %{time_starttransfer}s\nTotal: %{time_total}s" \
  -X POST http://localhost:8000/v1/completions \
  -d '{"model":"...","prompt":"Hello","max_tokens":50}'
```

### 8.2 vLLM / SGLang Startup Failures

**Symptom:** Server crashes on startup with cryptic errors about CUDA, shared
memory, or unsupported model architectures.

**Common Errors & Fixes:**

| Error | Likely Cause | Fix |
|-------|-------------|-----|
| `CUDA version mismatch` | PyTorch CUDA ≠ system CUDA | Match versions: `nvcc --version` vs `torch.version.cuda` |
| `Insufficient shared memory` | Docker defaults to 64MB `/dev/shm` | Add `--shm-size=1g` to `docker run` |
| `Model not supported` | Architecture not in vLLM's registry | Check vLLM docs for supported models; try `--trust-remote-code` |
| `OOM during model loading` | Model doesn't fit on available GPU(s) | Use `--quantization awq` or `--tensor-parallel-size N` |
| `TypeError` in tokenizer | Tokenizer has custom code | Use `--trust-remote-code` flag |

**General startup checklist:**
```bash
python -c "import torch; print(torch.version.cuda, torch.cuda.is_available())"
nvidia-smi  # Check GPU visibility and driver version
pip show vllm  # Check installed version
```

### 8.3 Structured Output Failures

**Symptom:** Model returns invalid JSON, violates the schema, or the serving
framework hangs when constrained generation is enabled.

**Root Causes:**
1. **Schema too complex.** Deeply nested schemas with many `oneOf`/`anyOf` branches
   cause exponential blowup in the grammar/FSM used for constrained decoding.
2. **Grammar not supported.** The serving framework doesn't support the schema feature
   you're using (e.g., regex patterns, recursive types).
3. **Conflicting constraints.** `max_tokens` is too low to generate a valid JSON
   object, so the output is truncated mid-JSON.

**Fix:** Keep schemas flat and simple. Test with `max_tokens` set generously.
For vLLM, use `guided_json` or `guided_regex`. For OpenAI-compatible APIs, use
`response_format={"type": "json_schema", "json_schema": ...}`. Always validate
the response:
```python
import json
try:
    result = json.loads(response.choices[0].message.content)
except json.JSONDecodeError as e:
    print(f"Invalid JSON: {e}")
```

---
## 9. General Diagnostic Toolkit

When you have no idea what's wrong, run this comprehensive check. It validates
the entire environment: GPU, packages, model loading, tokenizer, and inference.

In [ ]:
# Comprehensive environment diagnostic
def full_diagnostic(model_id=None):
    """
    Run a complete environment check. Optionally test loading a model.
    """
    print("🏥 Full System Diagnostic")
    print("=" * 60)

    # 1. Python version
    print(f"\n🐍 Python: {sys.version}")

    # 2. Key package versions
    print("\n📦 Package versions:")
    packages = [
        "torch", "transformers", "tokenizers", "accelerate",
        "bitsandbytes", "peft", "trl", "datasets",
        "vllm", "sglang", "faiss-gpu", "faiss-cpu",
        "sentence-transformers", "langchain", "openai",
    ]
    for pkg in packages:
        try:
            mod = __import__(pkg.replace("-", "_"))
            ver = getattr(mod, "__version__", "installed (version unknown)")
            print(f"  ✅ {pkg}: {ver}")
        except ImportError:
            print(f"  ⬚  {pkg}: not installed")

    # 3. GPU check
    print("\n🖥️  GPU status:")
    if torch and torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            total = props.total_mem / 1024**3
            print(f"  GPU {i}: {props.name} ({total:.1f} GB VRAM)")
        print(f"  CUDA version: {torch.version.cuda}")
        print(f"  cuDNN version: {torch.backends.cudnn.version()}")
    else:
        print("  ❌ No CUDA GPU available")
        if torch:
            print(f"  torch.version.cuda = {torch.version.cuda}")
            print(f"  torch built with CUDA: {torch.cuda._is_compiled()}")

    # 4. Model loading test (optional)
    if model_id:
        print(f"\n🤖 Model loading test: {model_id}")
        try:
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            print(f"  ✅ Tokenizer loaded. Vocab size: {tok.vocab_size}")
            print(f'  Chat template: {"yes" if tok.chat_template else "no"}')
            print(f"  Special tokens: {tok.special_tokens_map}")

            # Test tokenization round-trip
            test_text = "Hello, this is a test."
            ids = tok.encode(test_text)
            decoded = tok.decode(ids, skip_special_tokens=True)
            if decoded.strip() == test_text:
                print("  ✅ Tokenizer round-trip OK")
            else:
                print(f"  ⚠️  Round-trip mismatch: '{decoded}'")

        except Exception as e:
            print(f"  ❌ Tokenizer failed: {e}")

        try:
            from transformers import AutoModelForCausalLM
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=torch.float16 if torch else None,
                device_map="auto",
                trust_remote_code=True,
            )
            param_count = sum(p.numel() for p in model.parameters())
            print(f"  ✅ Model loaded. Parameters: {param_count/1e9:.2f}B")
            print(f"  Device: {next(model.parameters()).device}")

            # Quick inference test
            inputs = tok("Hello", return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=10)
            result = tok.decode(out[0], skip_special_tokens=True)
            print(f"  ✅ Inference OK: '{result[:60]}'")

            # Cleanup
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"  ❌ Model loading failed: {e}")

    print("\n" + "=" * 60)
    print("Diagnostic complete.")

# Run with no model (environment check only)
full_diagnostic()

# Or test a specific model:
# full_diagnostic("meta-llama/Llama-3.1-8B-Instruct")

---
## 10. Key Takeaways & Quick Reference

### 🧠 Mental Models for Debugging AI Systems

1. **Always check the simplest explanation first.** Most "model bugs" are data bugs
   or configuration bugs. The model is doing exactly what you told it to — just not
   what you meant.

2. **Print intermediate results.** Don't trust that data flows correctly through a
   pipeline. Print the actual input to the model. Print the actual retrieved chunks.
   Print the actual tokenized training sample.

3. **Reproduce with minimal examples.** Before debugging a 1000-sample eval failure,
   isolate one failing case and trace it end-to-end.

4. **Compare against a known baseline.** If finetuning made things worse, compare
   outputs with the base model. If RAG is hallucinating, test with the context
   hardcoded in the prompt. Isolate the broken component.

5. **VRAM is the universal constraint.** If you remember one number: a parameter
   in fp16 costs 2 bytes. 7B parameters = 14 GB. Everything follows from this.

### 📋 Quick Reference Table

| Symptom | Likely Cause | First Fix |
|---------|-------------|----------|
| `CUDA out of memory` | Model/batch too large for VRAM | Reduce batch size or use 4-bit quantization |
| `torch.cuda.is_available() = False` | No GPU runtime or wrong PyTorch | Change Colab runtime to GPU; reinstall PyTorch with CUDA |
| Slow generation (< 5 tok/s) | Model on CPU or KV cache disabled | Check `model.device`; ensure `use_cache=True` |
| Model outputs garbage | Wrong tokenizer or chat template | Load tokenizer from same model ID; check `chat_template` |
| RAG returns irrelevant chunks | Embedding mismatch or chunks too large | Verify embedding model matches; use 256-512 token chunks |
| Hallucination with RAG | Context not in prompt or truncated | Count tokens; put context before question |
| FAISS dimension error | Query/index embedding dim mismatch | Check `index.d` vs query shape |
| Agent loops forever | No stop condition or tool always fails | Set `max_iterations`; add fallback logic |
| Tool call JSON error | LLM outputs malformed JSON | Use structured output API; strip markdown fences |
| Training loss flat | LR wrong or data format broken | Print decoded training samples; adjust LR |
| Loss spikes | Gradient explosion or bad data | Enable grad clipping; inspect high-loss batches |
| Finetuned model worse | Catastrophic forgetting or wrong eval | Use LoRA; match eval prompt format to training |
| Eval scores unreliable | Too few samples | Use 100+ samples; report confidence intervals |
| LLM judge inconsistent | Temperature > 0 or vague rubric | Set `temperature=0`; define score levels explicitly |
| Prompt not followed | Wrong format or too complex | Use model's chat template; simplify to one instruction |
| Output truncated | Prompt + output > context window | Count tokens first; shorten prompt or raise limit |
| vLLM won't start | CUDA version mismatch or OOM | Match CUDA versions; use quantization or tensor parallel |
| Invalid structured output | Schema too complex or low `max_tokens` | Flatten schema; increase token budget |

### 🔧 The Debugging Checklist

When something breaks, work through this list top-to-bottom:

- [ ] **Read the error message.** The answer is often right there.
- [ ] **Check GPU status.** `torch.cuda.is_available()`, `nvidia-smi`.
- [ ] **Check VRAM.** Is the model loaded? How much memory is free?
- [ ] **Print the actual input.** What does the model actually see?
- [ ] **Test with a trivial example.** Does `"Hello"` produce any output?
- [ ] **Check package versions.** `transformers`, `torch`, `vllm` — are they compatible?
- [ ] **Compare with baseline.** Does the base model / default config work?
- [ ] **Search the error message.** GitHub Issues and HuggingFace forums often have the fix.
- [ ] **Restart the runtime.** Stale state causes more bugs than anyone admits.

---

*This playbook is a living document. As you encounter new failure modes in the
course, add them here. The best debugging reference is the one you build yourself
from real failures.*

**Navigation:** [Back to top](#-debugging-playbook--common-failures--fixes-across-the-ai-stack) · 
[Systems README](./README.md)